In [ ]:
import anndata
import scanpy as sc
import os
import matplotlib.pyplot as plt
import omicverse as ov
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline
plt.rcParams.update({'font.size': 12})

plt.rcParams['pdf.fonttype'] = 42  # Ensures that fonts are saved as text, not outlines
# Change the font to 'DejaVu Sans' or another available font
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'

In [2]:
#####Change path and Load data
os.chdir("/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/data")
adata = anndata.read_h5ad("monkey_reanno_20250706.h5ad")

In [3]:
adata

AnnData object with n_obs × n_vars = 142570 × 44223
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'stage', 'percent.mt', 'species', 'embryo', 'platform', 'ann_level_2', 'ann_level_3', 'ann_level_1', 'doublet', 'doublet_score', 'scANVI_res_0.5', 'leiden_3', 'reanno', 'lineage'
    var: 'features', 'n_cells', 'highly_variable'
    uns: 'harmo.anno_colors', 'hvg', 'leiden', 'leiden_3_colors', 'lineage_colors', 'log1p', 'neighbors', 'orig.ident_colors', 'pca', 'reanno_colors', 'scANVI_res_0.5_colors', 'stage_colors', 'umap'
    obsm: 'X_pca', 'X_scANVI', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'logcounts'
    obsp: 'connectivities', 'distances'

In [ ]:
# plot
godsnot_102 = [
    "#FFFF00", "#1CE6FF", "#FF34FF", "#FF4A46", "#008941", "#006FA6", "#A30059",
    "#FFDBE5", "#7A4900", "#0000A6", "#63FFAC", "#B79762", "#004D43", "#8FB0FF",
    "#997D87", "#5A0007", "#809693", "#6A3A4C", "#1B4400", "#4FC601", "#3B5DFF",
    "#4A3B53", "#FF2F80", "#61615A", "#BA0900", "#6B7900", "#00C2A0", "#FFAA92",
    "#FF90C9", "#B903AA", "#D16100", "#DDEFFF", "#000035", "#7B4F4B", "#A1C299",
    "#300018", "#0AA6D8", "#013349", "#00846F", "#372101", "#FFB500", "#C2FFED",
    "#A079BF", "#CC0744", "#C0B9B2", "#C2FF99", "#001E09", "#00489C", "#6F0062",
    "#0CBD66", "#EEC3FF", "#456D75", "#B77B68", "#7A87A1", "#788D66", "#885578",
    "#FAD09F", "#FF8A9A", "#D157A0", "#BEC459", "#456648", "#0086ED", "#886F4C",
    "#34362D", "#B4A8BD", "#00A6AA", "#452C2C", "#636375", "#A3C8C9", "#FF913F",
    "#938A81", "#575329", "#00FECF", "#B05B6F", "#8CD0FF", "#3B9700", "#04F757",
    "#C8A1A1", "#1E6E00", "#7900D7", "#A77500", "#6367A9", "#A05837", "#6B002C",
    "#772600", "#D790FF", "#99ADC0", "#3A2465", "#922329", "#5B4534", "#FDE8DC",
    "#404E55", "#0089A3", "#CB7E98", "#A4E804", "#324E72"
]
sc.pl.umap(
    adata,
    color="reanno",
    palette=godsnot_102,  # 直接传入颜色列表
    frameon=False,
    wspace=0.6,
    save='monkey_reanno.pdf'  # 自动保存为 pdf
)

In [5]:
cell_type_to_genes = {
    'TE/TrB': ['KRT7', 'GATA2', 'CDX2', 'TEAD4', 'CSH1', 'HLA-G', 'MMP2', 'TFAP2A', 'DLX3', 'ACKR2', 'CYP19A1', 'ITGA6', 'GATA3', 'KRT18', 'TEAD3', 'MYBL2', 'TFAP2A', 'DLX6', 'GCM1', 'TFAP2A', 'DLX3', 'ACKR2', 'ITGA6'],
    'cytotrophoblasts (CTBs)': ['TEAD4', 'ITGA6', 'FABP5', 'PEG10', 'CDH1',],
    'syncytiotrophoblasts (STBs)': ['CGB2', 'CGB7', 'PSG3', 'PSG6',  'TBX3', 'CGA', 'PRR9', 'ANXA1', 'LGALS16', 'ATF3'],  #'TCL6',
    'extravillous cytotrophoblasts (EVTs)': ['HLA-G', 'CSH1', 'MMP2', 'ERBB2'],
    'haemato-endothelial progenitors (HEP)': ['PECAM1', 'MEF2C', 'RUNX1', 'GATA1', 'GP1BB', 'CD52', 'TAL1', 'PRSS57', 'MYB', 'CD34'], 
    'Haemogenic endothelial cells': ['SOX6', 'PECAM1', 'CDH5', 'KDR', 'TEK', 'BMP4', 'ETV2','TAL1' ],
    'Myeloid progenitors': ['CD36', 'CSF1R', 'LYVE1', 'PTPRC'],
    'Erythroid': ['HBZ',  'HBE1', 'HBM', 'GYPA'],  #'HBG1'
    'Megakaryocyte erythroid progenitors': ['GP1BB', 'ITGA2B', 'NFE2', 'GATA1'],
    'Erythro-Myeloid Progenitors': ['CSF1R', 'MYB','SPI1', 'PTPRC', 'CD52'],
    'primitive macrophage': ['CD14', 'CD68'],
    'primitive megakaryocyte': ['PF4', 'PPBP', 'CMTM5', 'GP9'],
    'primitive streak (PS)': ['TBXT', 'SP5', 'HOXA1', 'CDX1', 'CDX2', 'MIXL1', 'MESP1', 'HAND1', 'WNT5B'],
    'Gastrulating cells': ['TBXT', 'CDX1',  'NKX1-2'],
    'intermediate mesoderm': ['OSR1', 'EYA1'], 
    'Axial mesoderm': ['TBXT', 'MIXL1', 'HES7'], 
    'Emergent Mesoderm': ['MESP1', 'LHX1', 'OTX2', 'LEFTY2'],
    'paraxial mesoderm': ['MSGN1', 'HES7', 'CDX2'],
    'Advanced Mesoderm': ['GATA6', 'HAND1', 'BMP4', 'FOXF1'],
    'Nascent Mesoderm': ['TBXT', 'MESP1', 'SNAI1', 'TBX6', 'FGF8', 'FGF17'],
    'Lateral plate mesoderm': ['TBXT', 'MSGN1', 'CDX2',  'PRRX1', 'MSX1', 'BMP4', 'FOXC2', 'KDR', 'MYL7', 'HAPLN1', 'GATA6', 'LIX1', 'FOXF1'],
    'Somite': ['MEOX1', 'FOXC2', 'PAX9', 'NKX3-2'],
    'Cardiomyocytes': ['MYL7', 'TNNT2', 'NKX2-5', 'ACTA2'],
    'Presomitic mesoderm (PSM)': ['MESP2', 'MEOX1', 'RIPPLY1', 'RIPPLY2'],
    'Surface ectoderm': ['TFAP2A', 'DLX5'],
    'Neuromesodermal progenitor (NMP)': ['TBXT', 'CDX2', 'WNT3A', 'WNT5B', 'FGF8', 'FGF17', 'SOX2'],
    'Epiblast': ['POU5F1', 'NANOG', 'UTF1', 'FGF4', 'NODAL', 'DNMT3B', 'SOX2', 'DPPA5', 'MT1X', 'KHDC3L', 'MT1G', 'TDGF1', 'PRDM14'],
    'Epiblast(naive)': ['KHDC1L', 'POU5F1', 'NANOG', 'DPPA5'],
    'Epiblast(primed)': ['USP44'],  #'CD24'
    'hypoblast (HB)': ['GATA4', 'GATA6', 'PDGFRA', 'FN1', 'COL4A1', 'APOA1', 'IHH', 'OTX2', 'FOXA1', 'FOXA2', 'HNF4A', 'SOX17', 'PDGFRA', 'APOA1', 'S100A14', 'APOA2', 'CDH2', 'CPN1'],
    'extraembryonic mesoderm (ExM)': ['KDR', 'COL6A1', 'FLT1', 'COL3A1', 'NID2', 'POSTN', 'PITX1', 'LUM'],
    'visceral/yolk endoderm (VE/YE)': ['AFP', 'TTR', 'APOA4', 'APOC3', 'MTTP', 'MPC2', 'CKB'],
    'yolk sac endoderm (YSE)': ['AFP', 'TTR', 'GJB1'],
    'anterior visceral endoderm (AVE)': ['LHX1', 'HHEX', 'EOMES', 'GSC', 'CER1', 'LEFTY1', 'LEFTY2', 'NOG', 'OTX2', 'SHISA2'],
    'Amniotic ectoderm': ['WNT6', 'TFAP2A', 'BMP4', 'DLX5', 'TFAP2A', 'GATA3', 'VTCN1', 'GABRP', 'ISL1', 'HEY1'],
    'Notochord': ['NOTO', 'CDX2', 'CHRD'],
    'Connecting stalk': ['CDX2', 'CDH2', 'LCN15'],
    'Definitive endoderm': ['SOX17', 'CCKBR', 'FOXA2', 'ISL1'],
    'Primordial germ cells': ['NANOS3', 'DPPA5', 'NANOG', 'DND1', 'SOX17', 'LAMA4', 'KIT'],
    'Neural plate': ['SOX2', 'SOX3', 'PAX6'],
    'Forebrain progenitor': ['SIX3', 'RAX', 'LHX5', 'OTX2', 'LHX2', 'FEZF1', 'EMX2'],
    'Midbrain progenitor': ['EN1', 'FGF17', 'PAX7', 'PAX2'],
    'Hindbrain progenitor': ['EGR2', 'GBX2', 'HOXB3', 'HOXB7', 'MAFB', 'HOXA2', 'CRABP1'],
    'Spinal cord': ['HOXD1', 'PAX6', 'SOX2', 'NKX1-2', 'HOXA1', 'HOXA2', 'HNF1B','NEUROG2'], # 'HAGLR', 'HOTAIRM1'
    'Neural crest': ['PAX3', 'PAX7', 'FOXD3', 'SNAI2', 'SOX10', 'MPZ', 'SOX9'],
    'Gut': ['PDX1', 'NKX2-1', 'PPY', 'IRX1'],
    'foregut': ['NPY', 'PAX1', 'HHEX'],
    'midgut': ['MNX1', 'HOXB2', 'HOXC9'],
    'hindgut': ['LCN15',  'HOXA10', 'CDX2'],
    #'allantois': ['LCN15', 'OSR1', 'DCC']  
}

In [ ]:
for cell_type, genes in cell_type_to_genes.items():
    print(f"Dot plot for {cell_type}")
    sc.pl.dotplot(
        adata,
        var_names=genes,
        groupby="reanno",
        use_raw=False,
        standard_scale="var",  # Scale expression per gene (standardize across cells)
        title=cell_type,
        show=True
    )

In [7]:
cell_type_to_genes = {
    'TE/TrB': ['KRT7', 'GATA2', 'TFAP2A',],
    'cytotrophoblasts (CTBs)': ['TEAD4', 'ITGA6', 'CDH1',],
    'syncytiotrophoblasts (STBs)': [ 'PRR9','CGB2', 'PSG3',],
    'extravillous cytotrophoblasts (EVTs)': ['HLA-G', 'CSH1','ERBB2'],
    'Epiblast': ['POU5F1', 'KHDC3L', 'UTF1',],
    'Epiblast(naive)': ['KHDC1L', 'NANOG', 'DPPA5'],
    'Epiblast(primed)': ['USP44'],  #, 'CD24'
    'primitive streak (PS)': ['TBXT', 'SP5', 'MIXL1',],
    'Primordial germ cells': ['NANOS3', 'DPPA5','LAMA4', ],
    'Amniotic ectoderm': ['DLX5', 'VTCN1', 'GABRP'],
    'Neuromesodermal progenitor (NMP)': [ 'CDX2', 'WNT3A', 'WNT5B', ],
    'Neural crest': ['FOXD3',  'SOX10', 'MPZ',], 
    'Neural plate': ['SOX2', 'SOX3', 'PAX6'],  
    'Forebrain progenitor': ['SIX3', 'RAX', 'LHX5',],
    'Hindbrain progenitor': ['EGR2', 'MAFB', 'HOXA2'],
    'Midbrain progenitor': ['EN1', 'PAX7', 'PAX2'],
    'Spinal cord': ['HOXD1', 'PAX6',  'NEUROG2'],
    'Axial/Nascent mesoderm': ['SNAI1', 'TBX6', 'HES7'], 
    'Emergent Mesoderm': ['MESP1', 'LHX1', 'OTX2', ],
    'Presomitic mesoderm (PSM)': ['MESP2', 'RIPPLY1', 'RIPPLY2'],
    'Somite': ['MEOX1', 'PAX9', 'NKX3-2'],
    'Lateral plate mesoderm': [ 'BMP4', 'LIX1', 'MSX1'],
    'Cardiomyocytes': ['MYL7', 'TNNT2', 'NKX2-5'],
    'extraembryonic mesoderm (ExM)': [ 'COL6A1',  'COL3A1', 'LUM'],
    'Connecting stalk': ['CDX1', 'CDX2', 'LCN15'],    
    'hypoblast': ['GATA4', 'GATA6', 'PDGFRA', ],
    'anterior visceral endoderm (AVE)': ['LHX1',  'EOMES', 'GSC', ],
    'visceral/yolksac endoderm (VE/YE)': ['MTTP', 'APOC3', 'APOA4',],
    'yolk sac endoderm (YSE)': ['AFP', 'TTR', 'GJB1'],
    'Definitive endoderm': ['SOX17', 'CCKBR', 'FOXA2', 'ISL1'],
    'foregut': ['NPY', 'PAX1', 'HHEX'],
    'midgut': ['MNX1', 'HOXB2', 'HOXC9'],
    'hindgut': ['LCN15',  'HOXA10', 'CDX2'],
    'Notochord': ['NOTO', 'CDX2', 'CHRD'],
    'haemato-endothelial progenitors (HEP)': ['MEF2C', 'RUNX1','TAL1'], 
    'Endothelium': ['PECAM1', 'CDH5', 'KDR', ],
    'Erythroid': ['HBZ',  'HBE1'],  #'HBG1',
    'Megakaryocyte erythroid progenitors': ['GP1BB', 'ITGA2B', 'GATA1'],
    'primitive megakaryocyte': ['PF4', 'PPBP', 'CMTM5', 'GP9'],
    'Myeloid progenitors': ['CSF1R', 'LYVE1', 'PTPRC'],
    'primitive macrophage': ['CD14', 'CD68'],


    #'allantois': ['LCN15', 'OSR1', 'DCC']  
}

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=cell_type_to_genes,  # Dictionary mapping cell types to gene lists
    groupby="reanno",
    use_raw=False,
    standard_scale="var",  # Scale expression per gene
    title=cell_type,
  #  swap_axes=True,        # <<--- This makes genes on y-axis and groups on x-axis
    show=True,
    save='monkey_dotplot_markers.pdf'
)

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="reanno", method="wilcoxon",use_raw=False)

In [ ]:
# List to store DataFrames for each cluster
all_markers = []

for cluster in adata.obs["reanno"].cat.categories:
    # Get all markers for this cluster
    df = pd.DataFrame({
        'gene': adata.uns['rank_genes_groups']['names'][cluster],
        'pval': adata.uns['rank_genes_groups']['pvals'][cluster],
        'pval_adj': adata.uns['rank_genes_groups']['pvals_adj'][cluster],
        'logfoldchange': adata.uns['rank_genes_groups']['logfoldchanges'][cluster],
        'score': adata.uns['rank_genes_groups']['scores'][cluster]
    })
    df['cluster'] = cluster
    all_markers.append(df)

# Concatenate all cluster DataFrames
full_marker_df = pd.concat(all_markers, axis=0)

filtered = full_marker_df[(full_marker_df.pval_adj < 0.05) & (full_marker_df.logfoldchange > 1)]

# Save to CSV
filtered.to_csv("monkey_full_marker_gene_table.csv", index=False)


In [ ]:
sc.tl.dendrogram(adata, groupby="reanno")
sc.pl.rank_genes_groups_dotplot(adata, groupby="reanno", standard_scale="var", n_genes=5,use_raw=False, save='marker')

In [9]:
os.chdir("/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal")
import matplotlib.pyplot as plt
import scanpy as sc

# Define list of genes
genes_to_plot = [
    "POU5F1", "NANOG",  # epiblast
    "SOX2", "TTYH1",    # neural ectoderm
    "GATA3", "TFAP2A",
    "TBXT", "CDX1", "PDGFRA", "APOA2", "FOXA2", "NANOS3","TAL1", "ISL1","DLX5","ISL1",
    "PECAM1", "HBZ", "PTPRC", "HBM",
    "GABRP", "HEY1", "COL6A1", "COL6A2"
]

# Directory to save plots
output_dir = "./monkey_gene_expression_plots"
import os
os.makedirs(output_dir, exist_ok=True)

# Loop over each gene and plot/save individually
for gene in genes_to_plot:
    fig, ax = plt.subplots(figsize=(6, 5))  # Create a new figure for each gene
    sc.pl.umap(
        adata,
        color=gene,
        use_raw=False,
        cmap=sns.cubehelix_palette(dark=0, light=.9, as_cmap=True),
        ax=ax,
        show=False,  # Prevents Scanpy from calling plt.show()
        size=15
    )
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{gene}_umap.pdf", dpi=300, bbox_inches='tight')
    plt.close()

In [10]:
#define lineages
lineages = {
    'TE_TrB': [ 'TE', 'CTB_1','CTB_2','CTB_3','CTB_4', 'STB_1','STB_2', 'EVT_1','EVT_2', ],
    'epi': [  'Epiblast_1','Epiblast_2', 'Ectoderm_1','Ectoderm_2','Ectoderm_3','Ectoderm_4',],
    'Primitive.streak': ['Primitive.streak',],
    'NMP': ['Neuromesodermal.progenitor',],
    'Notochord': ['Notochord_1', 'Notochord_2',],
    'PGC': ['PGC',],
    'ExE_endo': ['Hypoblast','AVE','VE_YE','YS.endoderm_1','YS.endoderm_2',  ],
    'Amniotic_ecto': [ 'Amniotic.epi', 'Amniontic.ectoderm_1', 'Amniontic.ectoderm_2',],
    'neural_ecto': ['Neural.crest','Neural.ectoderm.fore_midbrain', 'Spinal.cord', ],
    'Endoderm': ['DE','Gut_1','Gut_2',],
    'meso_Exe.meso': ['Paraxial.mesoderm', 'Emergent.mesoderm','Pre-somatic.mesoderm', 'Somite','Rostral.mesoderm',
                  'Lateral.plate.mesoderm_1',  'Lateral.plate.mesoderm_2', 'Cardiac.mesoderm_1','Cardiac.mesoderm_2','Allantois', 
                  'Connecting.stalk','Amniotic.mesoderm', 'Exe.meso.progenitor_1', 'Exe.meso.progenitor_2',
                   'Pre-YS.mesoderm','YS.mesoderm',],
    'hemogenic': [ 'Hemogenic.endothelial.progenitor', 'Endothelium', 'Erythroid','Primitive.megakaryocyte', 'Myeloid.progenitor',  ],

}

# Create a new column 'lineage' with default values
adata.obs['lineage'] = 'Unknown'

# Loop through each lineage and assign the corresponding cells
for lineage, annotations in lineages.items():
    adata.obs.loc[adata.obs['reanno'].isin(annotations), 'lineage'] = lineage

# Save the updated AnnData object if needed
# adata.write('path_to_save/adata_with_lineage.h5ad')

# Verify the changes
adata.obs['lineage'].value_counts()


meso_Exe.meso       59781
TE_TrB              30114
epi                 11858
Amniotic_ecto        9677
hemogenic            9528
neural_ecto          6033
Endoderm             5834
ExE_endo             5742
NMP                  2027
PGC                  1075
Primitive.streak      510
Notochord             391
Name: lineage, dtype: int64

In [ ]:
# plot
sc.pl.umap(adata, color="lineage", save='monkey_lineage.pdf')

In [ ]:
# plot
sc.pl.umap(adata, color="stage", save='monkey_stage.pdf')

In [13]:
#rename dataset labels
# Rename 'anno' to 'level_1'
adata.obs.rename(columns={'ann_level_1': 'harmo.anno'}, inplace=True)
#adata.obs.rename(columns={'scANVI_res_0.5': 'ann_level_2'}, inplace=True)
adata.obs.rename(columns={'ann_level_2':'orig_anno' }, inplace=True)
adata.obs.rename(columns={'ann_level_3': 'orig_sub_anno'}, inplace=True)

In [14]:
##save dataset
adata.raw.var.rename(columns={'_index': 'index'}, inplace=True)
adata.write_h5ad(filename="./data/monkey_reanno_20250706.h5ad")